# Analyze AIT Log v2 Attack Phases

In [1]:
import pandas as pd

from utils import analyze_phase

## Load MSA Data

In [2]:
dataset = "aitv2"
scenario = "fox"
data_dir = f"../data/interim/{dataset}/{scenario}/flows_labeled"
file_path = f"{data_dir}/all_flows_labeled.csv"

In [3]:
df = pd.read_csv(file_path)

In [4]:
df.head()

,flow_id,start_time,end_time,duration,src_ip,sport,dst_ip,dport,proto,service,...,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,ip_proto,label,sensor_host,start_time_dt,end_time_dt,phase
0,f1921,1.642205e+09,1.642205e+09,0.099298,10.35.35.206,60538,91.189.91.157,123,udp,ntp,...,76,1,76,-,17,NTP,intranet_server,2022-01-15 00:00:00.869369984,2022-01-15 00:00:00.968667984,0
1,f14196,1.642205e+09,1.642205e+09,0.098276,192.168.128.4,60538,91.189.91.157,123,udp,ntp,...,76,1,76,-,17,NTP,inet_firewall,2022-01-15 00:00:00.870096922,2022-01-15 00:00:00.968372822,0
2,f3334,1.642205e+09,1.642205e+09,0.053324,172.17.129.140,43702,10.35.33.111,445,tcp,gssapi,...,514,4,379,-,6,benign_share,cloud_share,2022-01-15 00:00:02.130912066,2022-01-15 00:00:02.184236050,0
3,f5126,1.642205e+09,1.642205e+09,0.051889,172.17.129.140,43702,10.35.33.111,445,tcp,gssapi,...,514,4,379,-,6,benign_share,internal_share,2022-01-15 00:00:02.131725073,2022-01-15 00:00:02.183614016,0
4,f3335,1.642205e+09,1.642205e+09,0.012432,172.17.129.140,43704,10.35.33.111,445,tcp,gssapi,...,514,4,379,-,6,benign_share,cloud_share,2022-01-15 00:00:02.226455927,2022-01-15 00:00:02.238888025,0


In [5]:
df.value_counts("phase")

phase
0    400024
2     56443
1     14678
4        61
3        28
Name: count, dtype: int64

## Per Phase Analysis

In [6]:
df_benign = df[df["phase"] == 0]
df_phase_1 = df[df["phase"] == 1]
df_phase_2 = df[df["phase"] == 2]
df_phase_3 = df[df["phase"] == 3]
df_phase_4 = df[df["phase"] == 4]   

### Phase 1 - Data Exfiltration

In [7]:
phase_1_labels = df_phase_1["label"].value_counts()
print("Phase 1 Label Distribution:")
print(phase_1_labels)

Phase 1 Label Distribution:
label
data exfiltration    14678
Name: count, dtype: int64


In [8]:
analyze_phase(df_phase_1, "Phase 1")

Total Flows: 14678

 --- IP distribution ---

Source IPs (1):
src_ip
192.168.255.254    14678
Name: count, dtype: int64

Destination IPs (1):
dst_ip
192.168.130.77    14678
Name: count, dtype: int64

 --- Port distribution ---
Source Ports (13109):
sport
6123     4
50163    4
65507    4
35669    4
41516    3
        ..
53576    1
58676    1
23005    1
38452    1
43646    1
Name: count, Length: 13109, dtype: int64

Destination Ports (1):
dport
53    14678
Name: count, dtype: int64


(src_ip
 192.168.255.254    14678
 Name: count, dtype: int64,
 dst_ip
 192.168.130.77    14678
 Name: count, dtype: int64,
 sport
 6123     4
 50163    4
 65507    4
 35669    4
 41516    3
         ..
 53576    1
 58676    1
 23005    1
 38452    1
 43646    1
 Name: count, Length: 13109, dtype: int64,
 dport
 53    14678
 Name: count, dtype: int64)

In [9]:
def analyze_origin_destination(df, phase_name):
    origins = df["local_orig"].value_counts()
    print(f"{phase_name} Origin Distribution:")
    print(origins)

    destinations = df["local_resp"].value_counts()
    print(f"{phase_name} Destination Distribution:")
    print(destinations)

In [10]:
analyze_origin_destination(df_phase_1, "Phase 1")

Phase 1 Origin Distribution:
local_orig
T    14678
Name: count, dtype: int64
Phase 1 Destination Distribution:
local_resp
T    14678
Name: count, dtype: int64


### Phase 2 Analysis - Scanning

In [11]:
print("Number of samples in Phase 2:", len(df_phase_2))

Number of samples in Phase 2: 56443


In [12]:
analyze_origin_destination(df_phase_2, "Phase 2")

Phase 2 Origin Distribution:
local_orig
T    56443
Name: count, dtype: int64
Phase 2 Destination Distribution:
local_resp
T    56434
F        9
Name: count, dtype: int64


In [13]:
analyze_phase(df_phase_2, "Phase 2")

Total Flows: 56443

 --- IP distribution ---

Source IPs (5):
src_ip
172.17.130.196     52306
192.168.130.77      4128
192.168.128.4          5
192.168.128.195        3
192.168.130.110        1
Name: count, dtype: int64

Destination IPs (8204):
dst_ip
10.35.35.206       10666
172.17.130.37       2812
192.168.128.4       2649
10.35.33.111        2488
172.17.129.140      2393
                   ...  
192.168.129.212        2
10.35.35.118           2
192.168.128.170        2
10.35.32.1             1
172.17.128.63          1
Name: count, Length: 8204, dtype: int64

 --- Port distribution ---
Source Ports (13622):
sport
59174    17
59510    16
59944    16
51730    15
34510    15
         ..
41648     1
54762     1
39222     1
54742     1
37206     1
Name: count, Length: 13622, dtype: int64

Destination Ports (1413):
dport
443      24981
80       16526
587         45
993         21
25          20
         ...  
40000        7
53535        7
60123        6
823          6
1212         6
Name: 

(src_ip
 172.17.130.196     52306
 192.168.130.77      4128
 192.168.128.4          5
 192.168.128.195        3
 192.168.130.110        1
 Name: count, dtype: int64,
 dst_ip
 10.35.35.206       10666
 172.17.130.37       2812
 192.168.128.4       2649
 10.35.33.111        2488
 172.17.129.140      2393
                    ...  
 192.168.129.212        2
 10.35.35.118           2
 192.168.128.170        2
 10.35.32.1             1
 172.17.128.63          1
 Name: count, Length: 8204, dtype: int64,
 sport
 59174    17
 59510    16
 59944    16
 51730    15
 34510    15
          ..
 41648     1
 54762     1
 39222     1
 54742     1
 37206     1
 Name: count, Length: 13622, dtype: int64,
 dport
 443      24981
 80       16526
 587         45
 993         21
 25          20
          ...  
 40000        7
 53535        7
 60123        6
 823          6
 1212         6
 Name: count, Length: 1413, dtype: int64)

In [14]:
df_phase_2.columns

Index(['flow_id', 'start_time', 'end_time', 'duration', 'src_ip', 'sport',
       'dst_ip', 'dport', 'proto', 'service', 'orig_bytes', 'resp_bytes',
       'conn_state', 'local_orig', 'local_resp', 'missed_bytes', 'history',
       'orig_pkts', 'orig_ip_bytes', 'resp_pkts', 'resp_ip_bytes',
       'tunnel_parents', 'ip_proto', 'label', 'sensor_host', 'start_time_dt',
       'end_time_dt', 'phase'],
      dtype='object')

In [15]:
phase_2_labels = df_phase_2["label"].value_counts()
print("Phase 2 Label Distribution:")
print(phase_2_labels)

Phase 2 Label Distribution:
label
host_discover_local      32788
service_scan             13924
dirb_scan                 8259
dns_brute_force_start     1192
host_discover_dmz          152
wpscan                     128
Name: count, dtype: int64


In [16]:
phase_2_labels = df_phase_2["label"].unique()
print("Unique labels in Phase 2:", phase_2_labels)

Unique labels in Phase 2: ['dns_brute_force_start' 'host_discover_local' 'host_discover_dmz'
 'service_scan' 'wpscan' 'dirb_scan']


In [17]:
phase_2_protocols = df_phase_2["proto"].value_counts()
print("Phase 2 Protocol Distribution:")
print(phase_2_protocols)

Phase 2 Protocol Distribution:
proto
tcp    56443
Name: count, dtype: int64


In [18]:
phase_2_ports = df_phase_2["dport"].value_counts()
print("Phase 2 Destination Port Distribution:")
print(phase_2_ports)

Phase 2 Destination Port Distribution:
dport
443      24981
80       16526
587         45
993         21
25          20
         ...  
40000        7
53535        7
60123        6
823          6
1212         6
Name: count, Length: 1413, dtype: int64


In [19]:
# Label = host discover local
curr_label = phase_2_labels[0]
print(f"Analyzing Phase 2 - {curr_label}")
df_host_discover_local = df_phase_2[df_phase_2["label"] == curr_label]
analyze_origin_destination(df_host_discover_local, f"Phase 2 - {curr_label}")
analyze_phase(df_host_discover_local, f"Phase 2 - {curr_label}")

Analyzing Phase 2 - dns_brute_force_start
Phase 2 - dns_brute_force_start Origin Distribution:
local_orig
T    1192
Name: count, dtype: int64
Phase 2 - dns_brute_force_start Destination Distribution:
local_resp
T    1192
Name: count, dtype: int64
Total Flows: 1192

 --- IP distribution ---

Source IPs (1):
src_ip
172.17.130.196    1192
Name: count, dtype: int64

Destination IPs (1):
dst_ip
172.17.128.1    1192
Name: count, dtype: int64

 --- Port distribution ---
Source Ports (1149):
sport
56168    3
50802    2
57824    2
54470    2
60800    2
        ..
55468    1
35132    1
39152    1
39344    1
58562    1
Name: count, Length: 1149, dtype: int64

Destination Ports (1190):
dport
443      2
80       2
1027     1
8085     1
5968     1
        ..
1334     1
9643     1
2522     1
5999     1
49155    1
Name: count, Length: 1190, dtype: int64


(src_ip
 172.17.130.196    1192
 Name: count, dtype: int64,
 dst_ip
 172.17.128.1    1192
 Name: count, dtype: int64,
 sport
 56168    3
 50802    2
 57824    2
 54470    2
 60800    2
         ..
 55468    1
 35132    1
 39152    1
 39344    1
 58562    1
 Name: count, Length: 1149, dtype: int64,
 dport
 443      2
 80       2
 1027     1
 8085     1
 5968     1
         ..
 1334     1
 9643     1
 2522     1
 5999     1
 49155    1
 Name: count, Length: 1190, dtype: int64)

In [20]:
# Label = host discover local
df_host_discover_local = df_phase_2[df_phase_2["label"] == "host_discover_local"]
# analyze_origin_destination(df_host_discover_local, "Phase 2 - Host Discover Local")
analyze_phase(df_host_discover_local, "Phase 2 - Host Discover Local")

Total Flows: 32788

 --- IP distribution ---

Source IPs (1):
src_ip
172.17.130.196    32788
Name: count, dtype: int64

Destination IPs (8192):
dst_ip
10.35.35.206    30
10.35.35.204     7
10.35.35.202     6
10.35.33.144     5
10.35.62.78      4
                ..
10.35.33.116     2
10.35.32.164     2
10.35.33.111     2
10.35.35.118     2
10.35.32.1       1
Name: count, Length: 8192, dtype: int64

 --- Port distribution ---
Source Ports (12752):
sport
57950    11
54154    11
57970     9
44204     9
52666     9
         ..
57546     1
50638     1
42880     1
60334     1
50218     1
Name: count, Length: 12752, dtype: int64

Destination Ports (2):
dport
80     16396
443    16392
Name: count, dtype: int64


(src_ip
 172.17.130.196    32788
 Name: count, dtype: int64,
 dst_ip
 10.35.35.206    30
 10.35.35.204     7
 10.35.35.202     6
 10.35.33.144     5
 10.35.62.78      4
                 ..
 10.35.33.116     2
 10.35.32.164     2
 10.35.33.111     2
 10.35.35.118     2
 10.35.32.1       1
 Name: count, Length: 8192, dtype: int64,
 sport
 57950    11
 54154    11
 57970     9
 44204     9
 52666     9
          ..
 57546     1
 50638     1
 42880     1
 60334     1
 50218     1
 Name: count, Length: 12752, dtype: int64,
 dport
 80     16396
 443    16392
 Name: count, dtype: int64)

In [21]:
def attack_frequency(df, phase_name):
    time_range = df["end_time"].max() - df["start_time"].min()
    attack_count = len(df)
    frequency = attack_count / time_range if time_range > 0 else 0
    print(f"{phase_name} Attack Frequency: {frequency:.4f} attacks per second")

In [22]:
print(f"Analyzing Phase 2:\n")
for curr_label in phase_2_labels:
    print(f"=== {curr_label} ===")
    df_curr_label = df_phase_2[df_phase_2["label"] == curr_label]
    # analyze_origin_destination(df_curr_label, f"Phase 2 - {curr_label}")
    analyze_phase(df_curr_label, f"Phase 2 - {curr_label}")
    # attack_frequency(df_curr_label, f"Phase 2 - {curr_label}")
    print()

Analyzing Phase 2:

=== dns_brute_force_start ===
Total Flows: 1192

 --- IP distribution ---

Source IPs (1):
src_ip
172.17.130.196    1192
Name: count, dtype: int64

Destination IPs (1):
dst_ip
172.17.128.1    1192
Name: count, dtype: int64

 --- Port distribution ---
Source Ports (1149):
sport
56168    3
50802    2
57824    2
54470    2
60800    2
        ..
55468    1
35132    1
39152    1
39344    1
58562    1
Name: count, Length: 1149, dtype: int64

Destination Ports (1190):
dport
443      2
80       2
1027     1
8085     1
5968     1
        ..
1334     1
9643     1
2522     1
5999     1
49155    1
Name: count, Length: 1190, dtype: int64

=== host_discover_local ===
Total Flows: 32788

 --- IP distribution ---

Source IPs (1):
src_ip
172.17.130.196    32788
Name: count, dtype: int64

Destination IPs (8192):
dst_ip
10.35.35.206    30
10.35.35.204     7
10.35.35.202     6
10.35.33.144     5
10.35.62.78      4
                ..
10.35.33.116     2
10.35.32.164     2
10.35.33.111   

### Phase 3 Analysis - Exploitation

In [23]:
phase_3_labels = df_phase_3["label"].unique()
print("Unique labels in Phase 3:", phase_3_labels)

Unique labels in Phase 3: ['upload_rce_shell' 'check_user_id' 'check_netstat_t' 'read_resolv'
 'check_network_config' 'check_ps_a' 'check_release' 'read_group'
 'read_passwd' 'check_date' 'list_web_dir' 'check_wp_config'
 'dump_wp_users' 'read_profile']


In [24]:
for curr_label in phase_3_labels:
    print(f"Analyzing Phase 3 - {curr_label}")
    df_curr_label = df_phase_3[df_phase_3["label"] == curr_label]
    # analyze_origin_destination(df_curr_label, f"Phase 3 - {curr_label}")
    analyze_phase(df_curr_label, f"Phase 3 - {curr_label}")
    attack_frequency(df_curr_label, f"Phase 3 - {curr_label}")
    print(len(df_curr_label))
    print()

Analyzing Phase 3 - upload_rce_shell
Total Flows: 2

 --- IP distribution ---

Source IPs (1):
src_ip
172.17.130.196    2
Name: count, dtype: int64

Destination IPs (1):
dst_ip
10.35.35.206    2
Name: count, dtype: int64

 --- Port distribution ---
Source Ports (1):
sport
35148    2
Name: count, dtype: int64

Destination Ports (1):
dport
443    2
Name: count, dtype: int64
Phase 3 - upload_rce_shell Attack Frequency: 0.4971 attacks per second
2

Analyzing Phase 3 - check_user_id
Total Flows: 2

 --- IP distribution ---

Source IPs (1):
src_ip
172.17.130.196    2
Name: count, dtype: int64

Destination IPs (1):
dst_ip
10.35.35.206    2
Name: count, dtype: int64

 --- Port distribution ---
Source Ports (1):
sport
35150    2
Name: count, dtype: int64

Destination Ports (1):
dport
443    2
Name: count, dtype: int64
Phase 3 - check_user_id Attack Frequency: 4.6372 attacks per second
2

Analyzing Phase 3 - check_netstat_t
Total Flows: 2

 --- IP distribution ---

Source IPs (1):
src_ip
172.17.

In [25]:
analyze_origin_destination(df_phase_3, "Phase 3")

Phase 3 Origin Distribution:
local_orig
T    28
Name: count, dtype: int64
Phase 3 Destination Distribution:
local_resp
T    28
Name: count, dtype: int64


In [26]:
analyze_phase(df_phase_3, "Phase 3")

Total Flows: 28

 --- IP distribution ---

Source IPs (1):
src_ip
172.17.130.196    28
Name: count, dtype: int64

Destination IPs (1):
dst_ip
10.35.35.206    28
Name: count, dtype: int64

 --- Port distribution ---
Source Ports (14):
sport
35148    2
35150    2
35152    2
35154    2
35156    2
35158    2
35160    2
35162    2
35164    2
35166    2
35168    2
35170    2
35172    2
35174    2
Name: count, dtype: int64

Destination Ports (1):
dport
443    28
Name: count, dtype: int64


(src_ip
 172.17.130.196    28
 Name: count, dtype: int64,
 dst_ip
 10.35.35.206    28
 Name: count, dtype: int64,
 sport
 35148    2
 35150    2
 35152    2
 35154    2
 35156    2
 35158    2
 35160    2
 35162    2
 35164    2
 35166    2
 35168    2
 35170    2
 35172    2
 35174    2
 Name: count, dtype: int64,
 dport
 443    28
 Name: count, dtype: int64)

### Phase 4 Analysis - Cracking

In [27]:
analyze_origin_destination(df_phase_4, "Phase 4")

Phase 4 Origin Distribution:
local_orig
T    61
Name: count, dtype: int64
Phase 4 Destination Distribution:
local_resp
T    60
F     1
Name: count, dtype: int64


In [28]:
analyze_phase(df_phase_4, "Phase 4")

Total Flows: 61

 --- IP distribution ---

Source IPs (3):
src_ip
172.17.130.196    56
192.168.128.4      4
10.35.35.206       1
Name: count, dtype: int64

Destination IPs (5):
dst_ip
172.17.130.37      34
10.35.35.206       22
192.168.130.77      3
151.101.14.109      1
192.168.255.254     1
Name: count, dtype: int64

 --- Port distribution ---
Source Ports (31):
sport
55754    3
36406    2
35176    2
36408    2
36410    2
35178    2
51940    2
35180    2
36412    2
35182    2
59296    2
36414    2
36416    2
36418    2
36420    2
36422    2
36424    2
36426    2
36428    2
36430    2
36432    2
36434    2
36436    2
50330    2
50328    2
50334    2
50332    2
50336    2
50338    2
51972    1
45119    1
Name: count, dtype: int64

Destination Ports (3):
dport
443    43
80     17
53      1
Name: count, dtype: int64


(src_ip
 172.17.130.196    56
 192.168.128.4      4
 10.35.35.206       1
 Name: count, dtype: int64,
 dst_ip
 172.17.130.37      34
 10.35.35.206       22
 192.168.130.77      3
 151.101.14.109      1
 192.168.255.254     1
 Name: count, dtype: int64,
 sport
 55754    3
 36406    2
 35176    2
 36408    2
 36410    2
 35178    2
 51940    2
 35180    2
 36412    2
 35182    2
 59296    2
 36414    2
 36416    2
 36418    2
 36420    2
 36422    2
 36424    2
 36426    2
 36428    2
 36430    2
 36432    2
 36434    2
 36436    2
 50330    2
 50328    2
 50334    2
 50332    2
 50336    2
 50338    2
 51972    1
 45119    1
 Name: count, dtype: int64,
 dport
 443    43
 80     17
 53      1
 Name: count, dtype: int64)

In [29]:
df_phase_4_protocols = df_phase_4["proto"].value_counts()
print("Phase 4 Protocol Distribution:")
print(df_phase_4_protocols)

Phase 4 Protocol Distribution:
proto
tcp    61
Name: count, dtype: int64


In [30]:
analyze_origin_destination(df_benign, "Benign")

Benign Origin Distribution:
local_orig
T    400021
F         3
Name: count, dtype: int64
Benign Destination Distribution:
local_resp
T    337453
F     62571
Name: count, dtype: int64


In [31]:
analyze_origin_destination(df, "All Phases")

All Phases Origin Distribution:
local_orig
T    471231
F         3
Name: count, dtype: int64
All Phases Destination Distribution:
local_resp
T    408653
F     62581
Name: count, dtype: int64


In [32]:
phase4_ips = set(df_phase_4["src_ip"]).union(set(df_phase_4["dst_ip"]))
print("Unique IPs in Phase 4:", len(phase4_ips))
print(phase4_ips)

Unique IPs in Phase 4: 7
{'10.35.35.206', '192.168.130.77', '192.168.255.254', '172.17.130.37', '151.101.14.109', '172.17.130.196', '192.168.128.4'}


In [33]:
df_phase_4[df_phase_4["dst_ip"] == "10.35.35.206"]
# {'172.17.130.196', '172.17.130.37', '192.168.255.254', '10.35.35.206', '192.168.128.4', '151.101.14.109', '192.168.130.77'}

,flow_id,start_time,end_time,duration,src_ip,sport,dst_ip,dport,proto,service,...,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,ip_proto,label,sensor_host,start_time_dt,end_time_dt,phase
384659,f78497,1.642510e+09,1.642510e+09,32.502218,172.17.130.196,35176,10.35.35.206,443,tcp,ssl,...,7646,425,519402,-,6,online_cracking,vpn,2022-01-18 12:38:29.550966024,2022-01-18 12:39:02.053184032,4
384660,f17233,1.642510e+09,1.642510e+09,32.501220,172.17.130.196,35176,10.35.35.206,443,tcp,ssl,...,7646,433,529265,-,6,online_cracking,intranet_server,2022-01-18 12:38:29.551500082,2022-01-18 12:39:02.052720070,4
384704,f78499,1.642510e+09,1.642510e+09,1.099385,172.17.130.196,35178,10.35.35.206,443,tcp,ssl,...,10430,405,492169,-,6,online_cracking,vpn,2022-01-18 12:39:03.249042988,2022-01-18 12:39:04.348428011,4
384705,f17234,1.642510e+09,1.642510e+09,1.098563,172.17.130.196,35178,10.35.35.206,443,tcp,ssl,...,10430,436,530292,-,6,online_cracking,intranet_server,2022-01-18 12:39:03.249373913,2022-01-18 12:39:04.347936869,4
384709,f78502,1.642510e+09,1.642510e+09,2.725528,172.17.130.196,35180,10.35.35.206,443,tcp,ssl,...,10814,430,524001,-,6,online_cracking,vpn,2022-01-18 12:39:06.616677999,2022-01-18 12:39:09.342206001,4
384710,f17237,1.642510e+09,1.642510e+09,2.724900,172.17.130.196,35180,10.35.35.206,443,tcp,ssl,...,10814,434,529317,-,6,online_cracking,intranet_server,2022-01-18 12:39:06.616926908,2022-01-18 12:39:09.341826916,4
384724,f78528,1.642510e+09,1.642510e+09,0.091874,172.17.130.196,35182,10.35.35.206,443,tcp,ssl,...,8838,382,463247,-,6,online_cracking,vpn,2022-01-18 12:39:11.009594917,2022-01-18 12:39:11.101468801,4
384725,f17245,1.642510e+09,1.642510e+09,0.091646,172.17.130.196,35182,10.35.35.206,443,tcp,ssl,...,8838,402,489827,-,6,online_cracking,intranet_server,2022-01-18 12:39:11.009860992,2022-01-18 12:39:11.101506948,4
384741,f78507,1.642510e+09,1.642510e+09,96.252340,172.17.130.196,59296,10.35.35.206,80,tcp,http,...,1163,10,6644,-,6,online_cracking,vpn,2022-01-18 12:39:45.911967993,2022-01-18 12:41:22.164308071,4
384742,f17242,1.642510e+09,1.642510e+09,96.250871,172.17.130.196,59296,10.35.35.206,80,tcp,http,...,1163,10,6644,-,6,online_cracking,intranet_server,2022-01-18 12:39:45.912630081,2022-01-18 12:41:22.163501024,4


In [34]:
df_phase_4[df_phase_4["src_ip"] == "10.35.35.206"]

,flow_id,start_time,end_time,duration,src_ip,sport,dst_ip,dport,proto,service,...,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,ip_proto,label,sensor_host,start_time_dt,end_time_dt,phase
384711,f17236,1.642510e+09,1.642510e+09,2.582308,10.35.35.206,55754,192.168.130.77,80,tcp,http,...,354020,189,177269,-,6,online_cracking,intranet_server,2022-01-18 12:39:06.746751070,2022-01-18 12:39:09.329059124,4


In [35]:
def get_phase_ips(df, phase):
    print(f"=== Analyzing Phase {phase} IPs ===")
    phase_ips = set(df["src_ip"].unique()) | set(df["dst_ip"].unique())
    phase_src_ips = set(df["src_ip"].unique())
    phase_dst_ips = set(df["dst_ip"].unique())

    print(f"Unique IPs in Phase {phase}:", phase_ips)
    print(f"Unique Source IPs in Phase {phase}:", phase_src_ips)
    print(f"Unique Destination IPs in Phase {phase}:", phase_dst_ips)
    print()

    return phase_ips, phase_src_ips, phase_dst_ips

In [36]:
phase1_ips, phase1_src_ips, phase1_dst_ips = get_phase_ips(df_phase_1, 1)
phase2_ips, phase2_src_ips, phase2_dst_ips = get_phase_ips(df_phase_2, 2)
phase3_ips, phase3_src_ips, phase3_dst_ips = get_phase_ips(df_phase_3, 3)
phase4_ips, phase4_src_ips, phase4_dst_ips = get_phase_ips(df_phase_4, 4)

=== Analyzing Phase 1 IPs ===
Unique IPs in Phase 1: {'192.168.130.77', '192.168.255.254'}
Unique Source IPs in Phase 1: {'192.168.255.254'}
Unique Destination IPs in Phase 1: {'192.168.130.77'}

=== Analyzing Phase 2 IPs ===


Unique IPs in Phase 2: {'10.35.34.130', '10.35.56.113', '10.35.42.208', '10.35.32.108', '10.35.63.22', '10.35.59.146', '10.35.60.86', '10.35.48.6', '10.35.59.55', '10.35.36.141', '10.35.54.22', '10.35.59.102', '10.35.37.88', '10.35.52.99', '10.35.45.92', '10.35.46.112', '10.35.53.161', '10.35.34.21', '10.35.44.177', '10.35.56.62', '10.35.41.239', '10.35.51.57', '10.35.55.97', '10.35.45.179', '10.35.44.101', '10.35.37.246', '10.35.37.185', '10.35.62.53', '10.35.42.125', '10.35.54.58', '10.35.35.113', '10.35.43.192', '10.35.40.110', '10.35.49.168', '10.35.39.148', '10.35.61.204', '10.35.53.68', '192.168.128.195', '10.35.48.2', '10.35.50.98', '10.35.39.187', '10.35.46.23', '10.35.61.169', '10.35.43.25', '10.35.50.248', '10.35.36.120', '10.35.41.172', '10.35.56.175', '10.35.58.100', '10.35.32.101', '10.35.56.121', '10.35.63.197', '10.35.63.48', '10.35.59.59', '10.35.38.214', '10.35.60.151', '10.35.63.201', '10.35.55.195', '10.35.46.34', '10.35.50.164', '10.35.63.238', '10.35.52.35', '10.35

In [37]:
all_phase_ips = phase1_ips | phase2_ips | phase3_ips | phase4_ips
print(len(all_phase_ips), "unique IPs across all phases.")

for i, ip in enumerate(all_phase_ips):
    in_phases = []
    if ip in phase1_ips:
        in_phases.append("Phase 1")
    if ip in phase2_ips:
        in_phases.append("Phase 2")
    if ip in phase3_ips:
        in_phases.append("Phase 3")
    if ip in phase4_ips:
        in_phases.append("Phase 4")
    
    if len(in_phases) > 1:
        print(f"IP {ip} appears in: {', '.join(in_phases)}")

8209 unique IPs across all phases.
IP 10.35.35.206 appears in: Phase 2, Phase 3, Phase 4
IP 172.17.130.37 appears in: Phase 2, Phase 4
IP 192.168.255.254 appears in: Phase 1, Phase 4
IP 192.168.128.4 appears in: Phase 2, Phase 4
IP 172.17.130.196 appears in: Phase 2, Phase 3, Phase 4
IP 192.168.130.77 appears in: Phase 1, Phase 2, Phase 4
